# Fingrid Data Explorer
Fetch and visualise real-time production and FCR-N price data from the Fingrid Open Data API.

**Requires:** `FINGRID_API_KEY` in `.env` (free at data.fingrid.fi)

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

from datetime import datetime, timezone, timedelta
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data_ingestion.fingrid import FingridClient

client = FingridClient()
print('API key loaded:', bool(client.api_key))

In [ ]:
# ── Date range ────────────────────────────────────────────────────────────────
# Adjust START and END to any historical window you want to inspect.
END   = datetime.now(tz=timezone.utc).replace(minute=0, second=0, microsecond=0)
START = END - timedelta(days=7)

## Nuclear production (dataset 188)
All Finnish nuclear capacity: Loviisa 1+2 (~507 MW each) + Olkiluoto 1+2+3.

In [ ]:
nuclear = client.get_nuclear_production(START, END)
nuclear_h = nuclear.resample('h').mean()  # downsample 3-min → hourly

print(f'Records: {len(nuclear)}  |  Hourly avg: {nuclear_h.mean():.0f} MW  '
      f'|  Range: {nuclear_h.min():.0f}–{nuclear_h.max():.0f} MW')
nuclear_h.tail()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=nuclear_h.index, y=nuclear_h.values,
                         mode='lines', name='Nuclear (MW)', line=dict(color='steelblue')))
fig.update_layout(title='Finnish nuclear production (hourly avg)',
                  xaxis_title='UTC', yaxis_title='MW',
                  hovermode='x unified', height=350)
fig.show()

## Hydro production (dataset 191)
Total Finnish hydro; includes Kemijoki, Vuoksi, Oulujoki and smaller rivers.

In [ ]:
import time
time.sleep(2)  # respect Fingrid rate limit between calls

hydro = client.get_hydro_production(START, END)
hydro_h = hydro.resample('h').mean()

print(f'Records: {len(hydro)}  |  Hourly avg: {hydro_h.mean():.0f} MW  '
      f'|  Range: {hydro_h.min():.0f}–{hydro_h.max():.0f} MW')
hydro_h.tail()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=hydro_h.index, y=hydro_h.values,
                         mode='lines', name='Hydro (MW)', line=dict(color='teal')))
fig.update_layout(title='Finnish hydro production (hourly avg)',
                  xaxis_title='UTC', yaxis_title='MW',
                  hovermode='x unified', height=350)
fig.show()

## FCR-N capacity prices (dataset 317)
Hourly clearing prices for Frequency Containment Reserve for Normal operation (EUR/MW/h).
This is the market the hydro flexibility is offered into.

In [ ]:
time.sleep(2)

fcr_n = client.get_fcr_n_prices(START, END)

if len(fcr_n):
    print(f'Records: {len(fcr_n)}  |  Mean: {fcr_n.mean():.2f} EUR/MW/h  '
          f'|  Range: {fcr_n.min():.2f}–{fcr_n.max():.2f}')
    fcr_n.tail()
else:
    print('No FCR-N price data returned for this window (may not be published yet).')

In [ ]:
if len(fcr_n):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=fcr_n.index, y=fcr_n.values,
                             mode='lines+markers', name='FCR-N (EUR/MW/h)',
                             line=dict(color='darkorange'), marker=dict(size=4)))
    fig.update_layout(title='FCR-N capacity prices',
                      xaxis_title='UTC', yaxis_title='EUR/MW/h',
                      hovermode='x unified', height=350)
    fig.show()

## Combined view — production stack + FCR-N price

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Production (MW)', 'FCR-N price (EUR/MW/h)'),
                    vertical_spacing=0.08)

fig.add_trace(go.Scatter(x=nuclear_h.index, y=nuclear_h.values,
                         name='Nuclear', line=dict(color='steelblue')), row=1, col=1)
fig.add_trace(go.Scatter(x=hydro_h.index, y=hydro_h.values,
                         name='Hydro', line=dict(color='teal')), row=1, col=1)

if len(fcr_n):
    fig.add_trace(go.Scatter(x=fcr_n.index, y=fcr_n.values,
                             name='FCR-N', line=dict(color='darkorange')), row=2, col=1)

fig.update_layout(title='Finnish power system — last 7 days',
                  hovermode='x unified', height=550)
fig.show()

## Basic statistics

In [ ]:
stats = pd.DataFrame({
    'Nuclear (MW)':    nuclear_h.describe(),
    'Hydro (MW)':      hydro_h.describe(),
    'FCR-N (€/MW/h)': fcr_n.describe() if len(fcr_n) else pd.Series(dtype=float),
}).round(1)
stats